In [3]:
from databricks.connect import DatabricksSession
from databricks.sdk.runtime import dbutils
from pyspark.sql import functions as F
from pyspark.sql import Window
from databricks.sdk.runtime import dbutils

In [4]:
spark = (
    DatabricksSession.builder
    .serverless()
    .profile("DEFAULT")
    .getOrCreate()
)

dbutils.widgets.text("catalog", "workspace")
catalog = dbutils.widgets.get("catalog")

/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


In [7]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {catalog}.gold
""")

spark.sql(f"""
    SHOW SCHEMAS IN {catalog}
""").show()

+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|            silver|
+------------------+



In [8]:
silver_customers = spark.table(f"{catalog}.silver.customers")

silver_customers.printSchema()
print("Silver customers: ", silver_customers.count())

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)

Silver customers:  99441


In [9]:
multiple_customer_ids = (
    silver_customers
    .groupBy("customer_unique_id")
    .agg(
        F.count("customer_id").alias("distinct_customer_ids"),
    )
    .filter(F.col("distinct_customer_ids") > 1)
)

print("Logical customers with multiple customer_id values: ", multiple_customer_ids.count())

Logical customers with multiple customer_id values:  2997


In [10]:
customer_location_profile = (
    silver_customers
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("customer_city").alias("distinct_cities"),
        F.countDistinct("customer_state").alias("distinct_states"),
    )
)

print(
    "Logical customers with multiple city values: ",
    customer_location_profile.filter(F.col("distinct_cities") > 1).count()
)

print(
    "Logical customers with multiple states values: ",
    customer_location_profile.filter(F.col("distinct_states") > 1).count()
)

Logical customers with multiple city values:  122
Logical customers with multiple states values:  39


In [12]:
changing_customers = (
    customer_location_profile
    .filter((F.col("distinct_cities") > 1) | (F.col("distinct_states") > 1))
    .select("customer_unique_id")
)

(
    silver_customers
    .join(changing_customers, on="customer_unique_id", how="inner")
    .select(
        "customer_unique_id",
        "customer_id",
        "customer_city",
        "customer_state",
    )
    .orderBy("customer_unique_id")
    .show(30, truncate=False)
)

+--------------------------------+--------------------------------+-------------------+--------------+
|customer_unique_id              |customer_id                     |customer_city      |customer_state|
+--------------------------------+--------------------------------+-------------------+--------------+
|0178b244a5c281fb2ade54038dd4b161|483468a56a54dbbbf8f2b2354cc8a729|guaratingueta      |SP            |
|0178b244a5c281fb2ade54038dd4b161|ea6ba2b2e27f7efea73bdeab7fd6e4a0|novo horizonte     |SP            |
|08fb46d35bb3ab4037202c23592d1259|ebc513ad3aed97e53c11e0e773c10152|sao paulo          |SP            |
|08fb46d35bb3ab4037202c23592d1259|caded193e8e47b8362864762a83db3c5|jundiai            |SP            |
|09d74edf20acb4f9523fb1cf19a18456|1b4d105211b13b833f020c713d4bb56f|belo horizonte     |MG            |
|09d74edf20acb4f9523fb1cf19a18456|a8876ecd91fd22bc8b420c9a4be69955|nova lima          |MG            |
|0ceb502fc33a2ad327b08288c5310e2e|282fbce48e4d2077aad602dd125c9225|viana 

In [13]:
silver_orders = spark.table(f"{catalog}.silver.orders")

silver_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)



In [16]:
customer_location_history = (
    silver_customers
    .join(changing_customers, on="customer_unique_id", how="inner")
    .join(silver_orders.select("customer_id", "order_id", "order_purchase_timestamp"), on="customer_id", how="left")
    .select(
        "customer_unique_id",
        "customer_id",
        "customer_city",
        "customer_state",
        "order_id",
        "order_purchase_timestamp"
    )
)

(
    customer_location_history
    .orderBy("customer_unique_id", "order_purchase_timestamp").show(30, truncate=False)
)

+--------------------------------+--------------------------------+-------------------+--------------+--------------------------------+------------------------+
|customer_unique_id              |customer_id                     |customer_city      |customer_state|order_id                        |order_purchase_timestamp|
+--------------------------------+--------------------------------+-------------------+--------------+--------------------------------+------------------------+
|0178b244a5c281fb2ade54038dd4b161|483468a56a54dbbbf8f2b2354cc8a729|guaratingueta      |SP            |a72698cb5818851e1f19815692cf256b|2017-05-10 20:04:09     |
|0178b244a5c281fb2ade54038dd4b161|ea6ba2b2e27f7efea73bdeab7fd6e4a0|novo horizonte     |SP            |94ca38619d7e64f94facac403a99682d|2018-07-28 13:13:00     |
|08fb46d35bb3ab4037202c23592d1259|ebc513ad3aed97e53c11e0e773c10152|sao paulo          |SP            |b240d7e249b17ae4e87e7fc8023378c0|2018-06-03 23:08:46     |
|08fb46d35bb3ab4037202c23592d1259|

In [18]:
customer_window = (
    Window
    .partitionBy("customer_unique_id")
    .orderBy(
        F.col("order_purchase_timestamp").desc(),
        F.col("customer_id").desc()
    )
)


latest_customer_records = (
    silver_customers
    .join(
        silver_orders.select("customer_id", "order_purchase_timestamp"),
        on="customer_id",
        how="left",
    )
    .withColumn(
        "rn",
        F.row_number().over(customer_window)
    )
    .filter(F.col("rn") == 1)
)

print("Latest logical customer records: ", latest_customer_records.count())

Latest logical customer records:  96096


In [19]:
print(
    "Latest customer records with no purchase timestamp: ",
    latest_customer_records
    .filter(F.col("order_purchase_timestamp").isNull())
    .count()
)

Latest customer records with no purchase timestamp:  8


In [20]:
rejected_orders = spark.table(f"{catalog}.silver.rejected_orders")

rejected_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)
 |-- rejection_reason: string (nullable = true)



In [22]:
null_timestamp_customers = (
    latest_customer_records
    .filter(F.col("order_purchase_timestamp").isNull())
    .select(
        "customer_unique_id",
        "customer_id"
    )
)

(
    null_timestamp_customers
    .join(
        rejected_orders.select(
            "customer_id",
            "order_id",
            "rejection_reason"
        ),
        on="customer_id",
        how="left"
    )
    .show(truncate=False)
)

+--------------------------------+--------------------------------+--------------------------------+--------------------------------------+
|customer_id                     |customer_unique_id              |order_id                        |rejection_reason                      |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------------+
|29f0540231702fda0cfdee0a310f11aa|1bd06a0c0df8b23dacfd3725d2dc0bb9|2ebdfc4f15f23b91474edf87475f108e|delivered_status_missing_delivery_date|
|dd1b84a7286eb4524d52af4256c0ba24|cce5e8188bf42ffb3bb5b18ff58f5965|ab7c89dc1bf4a1ead9d6ec1ec8968a84|delivered_status_missing_delivery_date|
|28c37425f1127d887d7337f284080a0f|175378436e2978be55b8f4316bce4811|20edc82cf5400ce95e1afacc25798b31|delivered_status_missing_delivery_date|
|cfda40ca8dd0a5d486a9635b611b398a|3bc508d482a402715be4d5cf4020cc81|e69f75a717d64fc5ecdfae42b2e8e086|delivered_status_missing_delivery_date|
|ec05a6d8558c6455f0c

In [25]:
dim_customer = (
    latest_customer_records
    .select(
        F.xxhash64("customer_unique_id").alias("customer_key"),
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )
)

dim_customer.printSchema()
print("dim_customer rows: ", dim_customer.count())

root
 |-- customer_key: long (nullable = false)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

dim_customer rows:  96096


In [26]:
dim_customer.show(10, truncate=False)

+--------------------+--------------------------------+------------------------+-----------------+--------------+
|customer_key        |customer_unique_id              |customer_zip_code_prefix|customer_city    |customer_state|
+--------------------+--------------------------------+------------------------+-----------------+--------------+
|417703405539887701  |0000f6ccb0745a6a4b88665a16c9f078|66812                   |belem            |PA            |
|-3469180410442505301|0004aac84e0df4da2b147fca70cf8255|18040                   |sorocaba         |SP            |
|-364628029729050498 |000d460961d6dbfa3ec6c9f5805769e1|01206                   |sao paulo        |SP            |
|-5880530323642331311|000ec5bff359e1c0ad76a81a45cb598f|18160                   |salto de pirapora|SP            |
|-6544934338134274888|000fbf0473c10fc1ab6f8d2d286ce20c|13330                   |indaiatuba       |SP            |
|-9049562769005150846|0010a452c6d13139e50b57f19f52e04e|95611                   |taquara 

In [28]:
duplicate_customer_keys = (
    dim_customer
    .groupBy("customer_key")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate customer_key values: ",
    duplicate_customer_keys.count()
)

print(
    "Rows using customer_key 0: ",
    dim_customer
    .filter(F.col("customer_key") == 0)
    .count()
)

Duplicate customer_key values:  0
Rows using customer_key 0:  0


In [31]:
unknown_customer = (
    spark.range(1)
    .select(
        F.lit(0).cast("long").alias("customer_key"),
        F.lit("UNKNOWN").alias("customer_unique_id"),
        F.lit("00000").alias("customer_zip_code_prefix"),
        F.lit("unknown").alias("customer_city"),
        F.lit("UN").alias("customer_state")
    )
)

dim_customer_with_unknown = (
    dim_customer
    .unionByName(unknown_customer)
)

print(
    "dim_customer rows including unknown:",
    dim_customer_with_unknown.count()
)

dim_customer rows including unknown: 96097


In [32]:
dim_customer_with_unknown.filter(
    F.col("customer_key") == 0
).show(truncate=False)

+------------+------------------+------------------------+-------------+--------------+
|customer_key|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+------------+------------------+------------------------+-------------+--------------+
|0           |UNKNOWN           |00000                   |unknown      |UN            |
+------------+------------------+------------------------+-------------+--------------+



In [33]:
(
    dim_customer_with_unknown.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.gold.dim_customer")
)

print("Persisted dim_customer: ", spark.table(f"{catalog}.gold.dim_customer").count())

Persisted dim_customer:  96097


In [34]:
silver_products = spark.table(f"{catalog}.silver.products")

silver_products.printSchema()
print("Silver products: ", silver_products.count())

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)

Silver products:  32951


In [35]:
dim_product = (
    silver_products
    .select(
        F.xxhash64("product_id").alias("product_key"),
        "product_id",
        "product_category_name",
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )
)

dim_product.printSchema()
print("dim_product rows:", dim_product.count())

root
 |-- product_key: long (nullable = false)
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

dim_product rows: 32951


In [36]:
duplicate_product_keys = (
    dim_product
    .groupBy("product_key")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate product_key values: ", duplicate_product_keys.count())

print("Rows using product_key 0: ", dim_product.filter(F.col("product_key") == 0).count())

Duplicate product_key values:  0
Rows using product_key 0:  0


In [39]:
unknown_product = (
    spark.range(1)
    .select(
        F.lit(0).cast("long").alias("product_key"),
        F.lit("UNKNOWN").alias("product_id"),
        F.lit("unknown").alias("product_category_name"),
        F.lit(None).cast("int").alias("product_name_length"),
        F.lit(None).cast("int").alias("product_description_length"),
        F.lit(None).cast("int").alias("product_photos_qty"),
        F.lit(None).cast("int").alias("product_weight_g"),
        F.lit(None).cast("int").alias("product_length_cm"),
        F.lit(None).cast("int").alias("product_height_cm"),
        F.lit(None).cast("int").alias("product_width_cm")
    )
)

dim_product_with_unknown = (
    dim_product
    .unionByName(unknown_product)
)

print(
    "dim_product rows including unknown:",
    dim_product_with_unknown.count()
)

dim_product rows including unknown: 32952


In [40]:
dim_product_with_unknown.filter(
    F.col("product_key") == 0
).show(truncate=False)

+-----------+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_key|product_id|product_category_name|product_name_length|product_description_length|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+-----------+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|0          |UNKNOWN   |unknown              |NULL               |NULL                      |NULL              |NULL            |NULL             |NULL             |NULL            |
+-----------+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



In [41]:
(
    dim_product_with_unknown.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.gold.dim_product")
)

print("Persisted dim_product: ", spark.table(f"{catalog}.gold.dim_product").count())

Persisted dim_product:  32952


In [44]:
date_range = (
    silver_orders
    .agg(
        F.min(F.to_date("order_purchase_timestamp")).alias("min_date"),
        F.max(F.to_date("order_purchase_timestamp")).alias("max_date")
    )
)

date_range.show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2016-09-04|2018-10-17|
+----------+----------+



In [47]:
dim_date = (
    spark.sql("""
        SELECT explode(
            sequence(
                to_date('2016-09-04'),
                to_date('2018-10-17'),
                interval 1 day
            )
        ) AS full_date
    """)
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.dayofmonth("full_date").alias("day_of_month"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.month("full_date").alias("month_number"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.quarter("full_date").alias("quarter"),
        F.year("full_date").alias("year"),
        F.when(
            F.dayofweek("full_date").isin(1, 7),
            True
        ).otherwise(False).alias("is_weekend")
    )
)

print("dim_date rows:", dim_date.count())

dim_date rows: 774


In [48]:
dim_date.show(10, truncate=False)

+--------+----------+------------+---------+------------+------------+----------+-------+----+----------+
|date_key|full_date |day_of_month|day_name |week_of_year|month_number|month_name|quarter|year|is_weekend|
+--------+----------+------------+---------+------------+------------+----------+-------+----+----------+
|20160904|2016-09-04|4           |Sunday   |35          |9           |September |3      |2016|true      |
|20160905|2016-09-05|5           |Monday   |36          |9           |September |3      |2016|false     |
|20160906|2016-09-06|6           |Tuesday  |36          |9           |September |3      |2016|false     |
|20160907|2016-09-07|7           |Wednesday|36          |9           |September |3      |2016|false     |
|20160908|2016-09-08|8           |Thursday |36          |9           |September |3      |2016|false     |
|20160909|2016-09-09|9           |Friday   |36          |9           |September |3      |2016|false     |
|20160910|2016-09-10|10          |Saturday |36

In [49]:
(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.gold.dim_date")
)

print("Persisted dim_date: ", spark.table(f"{catalog}.gold.dim_date").count())

Persisted dim_date:  774


In [51]:
silver_order_items = spark.table(f"{catalog}.silver.order_items")

silver_order_items.printSchema()
print("Silver order items: ", silver_order_items.count())

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)
 |-- item_total: decimal(13,2) (nullable = true)

Silver order items:  112642


In [52]:
fact_order_item_base = (
    silver_order_items
    .join(
        silver_orders.select(
            "order_id",
            "customer_id",
            "order_purchase_timestamp",
            "order_status",
            "delivery_days",
            "delivery_delay_days",
            "is_late"
        ),
        on="order_id",
        how="inner"
    )
    .join(
        silver_customers.select(
            "customer_id",
            "customer_unique_id"
        ),
        on="customer_id",
        how="inner"
    )
)

fact_order_item_base.printSchema()
print("fact_order_item_base rows:", fact_order_item_base.count())

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)
 |-- item_total: decimal(13,2) (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_status: string (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)
 |-- customer_unique_id: string (nullable = true)

fact_order_item_base rows: 112642


In [53]:
dim_customer_ref = (
    spark.table(f"{catalog}.gold.dim_customer")
    .select(
        "customer_key",
        "customer_unique_id",
    )
)

fact_with_customer_key = (
    fact_order_item_base
    .join(
        dim_customer_ref,
        on="customer_unique_id",
        how="left"
    )
    .withColumn(
        "customer_key",
        F.coalesce(
            F.col("customer_key"),
            F.lit(0).cast("long")
        )
    )
)

print("Rows after customer dimension join: ", fact_with_customer_key.count())

print(
    "Facts using unknown customer_key: ",
    fact_with_customer_key
    .filter(F.col("customer_key") == 0)
    .count()
)

Rows after customer dimension join:  112642
Facts using unknown customer_key:  0


In [54]:
dim_product_ref = (
    spark.table(f"{catalog}.gold.dim_product")
    .select(
        "product_key",
        "product_id"
    )
)

fact_with_product_key = (
    fact_with_customer_key
    .join(
        dim_product_ref,
        on="product_id",
        how="left"
    )
    .withColumn(
        "product_key",
        F.coalesce(
            F.col("product_key"),
            F.lit(0).cast("long")
        )
    )
)

print("Rows after product dimension join:", fact_with_product_key.count())

print(
    "Facts using unknown product_key:",
    fact_with_product_key
    .filter(F.col("product_key") == 0)
    .count()
)

Rows after product dimension join: 112642
Facts using unknown product_key: 0


In [55]:
dim_date_ref = (
    spark.table(f"{catalog}.gold.dim_date")
    .select(
        "date_key",
        "full_date"
    )
)

fact_with_date_key = (
    fact_with_product_key
    .withColumn(
        "order_date",
        F.to_date("order_purchase_timestamp")
    )
    .join(
        dim_date_ref,
        F.col("order_date") == F.col("full_date"),
        how="left"
    )
)

print("Rows after date dimension join:", fact_with_date_key.count())

print(
    "Facts with missing date_key:",
    fact_with_date_key
    .filter(F.col("date_key").isNull())
    .count()
)

Rows after date dimension join: 112642
Facts with missing date_key: 0


In [56]:
fact_order_item = (
    fact_with_date_key
    .select(
        "order_id",
        "order_item_id",
        "customer_key",
        "product_key",
        "date_key",
        "seller_id",
        "price",
        "freight_value",
        "item_total",
        "order_status",
        "delivery_days",
        "delivery_delay_days",
        "is_late"
    )
)

fact_order_item.printSchema()
print("fact_order_item rows:", fact_order_item.count())

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- customer_key: long (nullable = false)
 |-- product_key: long (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- item_total: decimal(13,2) (nullable = true)
 |-- order_status: string (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)

fact_order_item rows: 112642


In [57]:
duplicate_fact_keys = (
    fact_order_item
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate fact grain keys: ", duplicate_fact_keys.count())

Duplicate fact grain keys:  0


In [58]:
fact_dq_summary = fact_order_item.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("customer_key").isNull(), 1).otherwise(0)
    ).alias("null_customer_key"),

    F.sum(
        F.when(F.col("product_key").isNull(), 1).otherwise(0)
    ).alias("null_product_key"),

    F.sum(
        F.when(F.col("date_key").isNull(), 1).otherwise(0)
    ).alias("null_date_key"),

    F.sum(
        F.when(F.col("price") < 0, 1).otherwise(0)
    ).alias("negative_price"),

    F.sum(
        F.when(F.col("freight_value") < 0, 1).otherwise(0)
    ).alias("negative_freight"),

    F.sum(
        F.when(F.col("item_total") < 0, 1).otherwise(0)
    ).alias("negative_item_total")
)

print(fact_dq_summary.show(truncate=False))

+----------+-----------------+----------------+-------------+--------------+----------------+-------------------+
|total_rows|null_customer_key|null_product_key|null_date_key|negative_price|negative_freight|negative_item_total|
+----------+-----------------+----------------+-------------+--------------+----------------+-------------------+
|112642    |0                |0               |0            |0             |0               |0                  |
+----------+-----------------+----------------+-------------+--------------+----------------+-------------------+

None


In [59]:
(
    fact_order_item.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.gold.fact_order_item")
)

print("Persisted fact_order_item: ", spark.table(f"{catalog}.gold.fact_order_item").count())

Persisted fact_order_item:  112642


In [60]:
gold_tables = [
    f"{catalog}.gold.dim_customer",
    f"{catalog}.gold.dim_product",
    f"{catalog}.gold.dim_date",
    f"{catalog}.gold.fact_order_item",
]

for table in gold_tables:
    print(f"{table}: {spark.table(table).count()}")

workspace.gold.dim_customer: 96097
workspace.gold.dim_product: 32952
workspace.gold.dim_date: 774
workspace.gold.fact_order_item: 112642


In [62]:
incremental_split = (
    fact_order_item_base
    .agg(
        F.sum(
            F.when(
                F.col("order_purchase_timestamp") < F.lit("2018-01-01"),
                1
            ).otherwise(0)
        ).alias("initial_batch_rows"),

        F.sum(
            F.when(
                F.col("order_purchase_timestamp") >= F.lit("2018-01-01"),
                1
            ).otherwise(0)
        ).alias("incremental_batch_rows")
    )
)

incremental_split.show()

+------------------+----------------------+
|initial_batch_rows|incremental_batch_rows|
+------------------+----------------------+
|             51232|                 61410|
+------------------+----------------------+



In [63]:
watermark = "2018-01-01"

initial_fact_batch = (
    fact_with_date_key
    .filter(F.col("order_purchase_timestamp") < F.lit(watermark))
    .select(*fact_order_item.columns)
)

print("Initial fact batch: ", initial_fact_batch.count())

Initial fact batch:  51232


In [64]:
(
    initial_fact_batch.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.gold.fact_order_item_incremental_demo")
)

print("Initial target rows: ", spark.table(f"{catalog}.gold.fact_order_item_incremental_demo").count())

Initial target rows:  51232


In [65]:
incremental_fact_batch = (
    fact_with_date_key
    .filter(F.col("order_purchase_timestamp") >= F.lit(watermark))
    .select(*fact_order_item.columns)
)

print("Incremental fact batch: ", incremental_fact_batch.count())

Incremental fact batch:  61410


In [66]:
incremental_fact_batch.createOrReplaceTempView("incremental_fact_batch")

I0813 12:32:44.423402 9951305 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


In [67]:
spark.sql(f"""
    MERGE INTO {catalog}.gold.fact_order_item_incremental_demo AS target
    USING incremental_fact_batch AS source
    ON  target.order_id = source.order_id
    AND target.order_item_id = source.order_item_id

    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [68]:
print("Target rows after MERGE: ", spark.table(f"{catalog}.gold.fact_order_item_incremental_demo").count())

Target rows after MERGE:  112642


In [69]:
spark.sql(f"""
    MERGE INTO {catalog}.gold.fact_order_item_incremental_demo AS target
    USING incremental_fact_batch AS source
    ON  target.order_id = source.order_id
    AND target.order_item_id = source.order_item_id

    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [70]:
print("Target rows after rerun: ", spark.table(f"{catalog}.gold.fact_order_item_incremental_demo").count())

Target rows after rerun:  112642


In [72]:
spark.sql(f"""
    DESCRIBE HISTORY {catalog}.gold.fact_order_item_incremental_demo
""").select(
    "version",
    "operation",
    "operationMetrics"
).show(5, truncate=False)

+-------+---------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation                        |operationMetrics                                                                                                                                                                                                                 

In [73]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_product_type1_demo
    AS
    SELECT *
    FROM {catalog}.gold.dim_product
""")

print("Type 1 demo rows:",spark.table(f"{catalog}.gold.dim_product_type1_demo").count())

Type 1 demo rows: 32952


In [75]:
type1_product = (
    spark.table(f"{catalog}.gold.dim_product_type1_demo")
    .filter(F.col("product_key") != 0)
    .orderBy("product_id")
    .limit(1)
)

type1_product.select(
    "product_key",
    "product_id",
    "product_category_name"
).show(truncate=False)

+--------------------+--------------------------------+---------------------+
|product_key         |product_id                      |product_category_name|
+--------------------+--------------------------------+---------------------+
|-6126185559987931235|00066f42aeeb9f3007548bb9d3f33c38|perfumaria           |
+--------------------+--------------------------------+---------------------+



In [77]:
type1_update = (
    type1_product
    .select("product_id")
    .withColumn(
        "product_category_name",
        F.lit("fragrances")
    )
)

type1_update.show(truncate=False)

+--------------------------------+---------------------+
|product_id                      |product_category_name|
+--------------------------------+---------------------+
|00066f42aeeb9f3007548bb9d3f33c38|fragrances           |
+--------------------------------+---------------------+



In [78]:
type1_update.createOrReplaceTempView("type1_product_update")

In [79]:
spark.sql(f"""
    MERGE INTO {catalog}.gold.dim_product_type1_demo AS target
    USING type1_product_update AS source
    ON target.product_id = source.product_id

    WHEN MATCHED THEN
        UPDATE SET
            target.product_category_name = source.product_category_name
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [81]:
(
    spark.table(f"{catalog}.gold.dim_product_type1_demo")
    .filter(F.col("product_id") == "00066f42aeeb9f3007548bb9d3f33c38")
    .select(
        "product_key",
        "product_id",
        "product_category_name"
    )
    .show(truncate=False)
)

+--------------------+--------------------------------+---------------------+
|product_key         |product_id                      |product_category_name|
+--------------------+--------------------------------+---------------------+
|-6126185559987931235|00066f42aeeb9f3007548bb9d3f33c38|fragrances           |
+--------------------+--------------------------------+---------------------+



In [83]:
type2_customer_source = (
    customer_location_history
    .filter(F.col("customer_unique_id") == "0178b244a5c281fb2ade54038dd4b161")
    .select(
        "customer_unique_id",
        "customer_id",
        "customer_city",
        "customer_state",
        "order_purchase_timestamp"
    )
    .orderBy("order_purchase_timestamp")
)

type2_customer_source.show(truncate=False)

+--------------------------------+--------------------------------+--------------+--------------+------------------------+
|customer_unique_id              |customer_id                     |customer_city |customer_state|order_purchase_timestamp|
+--------------------------------+--------------------------------+--------------+--------------+------------------------+
|0178b244a5c281fb2ade54038dd4b161|483468a56a54dbbbf8f2b2354cc8a729|guaratingueta |SP            |2017-05-10 20:04:09     |
|0178b244a5c281fb2ade54038dd4b161|ea6ba2b2e27f7efea73bdeab7fd6e4a0|novo horizonte|SP            |2018-07-28 13:13:00     |
+--------------------------------+--------------------------------+--------------+--------------+------------------------+



In [85]:
type2_window = (
    Window
    .partitionBy("customer_unique_id")
    .orderBy("order_purchase_timestamp")
)

type2_customer_history = (
    type2_customer_source
    .withColumn(
        "effective_from",
        F.col("order_purchase_timestamp")
    )
    .withColumn(
        "effective_to",
        F.lead("order_purchase_timestamp").over(type2_window)
    )
    .withColumn(
        "is_current",
        F.col("effective_to").isNull()
    )
    .withColumn(
        "customer_key",
        F.xxhash64(
            "customer_unique_id",
            "effective_from"
        )
    )
    .select(
        "customer_key",
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "effective_from",
        "effective_to",
        "is_current"
    )
)

type2_customer_history.show(truncate=False)

+--------------------+--------------------------------+--------------+--------------+-------------------+-------------------+----------+
|customer_key        |customer_unique_id              |customer_city |customer_state|effective_from     |effective_to       |is_current|
+--------------------+--------------------------------+--------------+--------------+-------------------+-------------------+----------+
|-6645923720405937658|0178b244a5c281fb2ade54038dd4b161|guaratingueta |SP            |2017-05-10 20:04:09|2018-07-28 13:13:00|false     |
|-4258461494087712859|0178b244a5c281fb2ade54038dd4b161|novo horizonte|SP            |2018-07-28 13:13:00|NULL               |true      |
+--------------------+--------------------------------+--------------+--------------+-------------------+-------------------+----------+



In [86]:
(
    type2_customer_history.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.gold.dim_customer_type2_demo")
)

print("Type 2 demo rows: ", spark.table(f"{catalog}.gold.dim_customer_type2_demo").count())

Type 2 demo rows:  2


In [88]:
type2_current_check = (
    spark.table(f"{catalog}.gold.dim_customer_type2_demo")
    .groupBy("customer_unique_id")
    .agg(
        F.sum(
            F.when(F.col("is_current") == True, 1).otherwise(0)
        ).alias("current_rows")
    )
)

type2_current_check.show(truncate=False)

+--------------------------------+------------+
|customer_unique_id              |current_rows|
+--------------------------------+------------+
|0178b244a5c281fb2ade54038dd4b161|1           |
+--------------------------------+------------+



In [89]:
invalid_type2_ranges = (
    spark.table(f"{catalog}.gold.dim_customer_type2_demo")
    .filter(
        F.col("effective_to").isNotNull()
        & (F.col("effective_to") <= F.col("effective_from"))
    )
)

print("Invalid Type 2 date ranges: ", invalid_type2_ranges.count())

Invalid Type 2 date ranges:  0


In [90]:
incremental_demo = spark.table(f"{catalog}.gold.fact_order_item_incremental_demo")

duplicate_incremental_facts = (
    incremental_demo
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate fact keys after incremental rerun: ", duplicate_incremental_facts.count())

Duplicate fact keys after incremental rerun:  0


In [91]:
full_fact = spark.table(f"{catalog}.gold.fact_order_item")

incremental_fact = spark.table(f"{catalog}.gold.fact_order_item_incremental_demo")

print("Rows in full fact but not incremental: ", full_fact.exceptAll(incremental_fact).count())

print("Rows in incremental but not full fact: ", incremental_fact.exceptAll(full_fact).count())

Rows in full fact but not incremental:  0
Rows in incremental but not full fact:  0


In [92]:
gold_validation = {
    "duplicate_customer_keys": (
        spark.table(f"{catalog}.gold.dim_customer")
        .groupBy("customer_key")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),

    "duplicate_product_keys": (
        spark.table(f"{catalog}.gold.dim_product")
        .groupBy("product_key")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),

    "duplicate_date_keys": (
        spark.table(f"{catalog}.gold.dim_date")
        .groupBy("date_key")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),

    "duplicate_fact_keys": (
        spark.table(f"{catalog}.gold.fact_order_item")
        .groupBy("order_id", "order_item_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),
}

for check, result in gold_validation.items():
    print(f"{check}: {result}")

duplicate_customer_keys: 0
duplicate_product_keys: 0
duplicate_date_keys: 0
duplicate_fact_keys: 0
